# MOD-07 · NB-06 — Environmental Epidemiology & Spatial Exposure Assessment
### Health Analytics with Python · Module 07: Geospatial & Spatial Epidemiology
---
**Learning objectives**
- Interpolate point-source pollution data to area-level estimates (kriging, IDW)
- Assess exposure-response relationships with ecological regression
- Apply air pollution exposure assessment methods (inverse distance, kriging)
- Conduct distributed lag non-linear models (DLNM) for time-series exposure
- Build environmental justice metrics: environmental burden by demographic group
- Address ecological fallacy in spatial health analyses

**Estimated time:** 3 hours | **Level:** Advanced  
**Libraries:** `numpy`, `scipy`, `sklearn`, `matplotlib`


## 1. Setup

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
from scipy.ndimage import gaussian_filter; from scipy import stats
warnings.filterwarnings("ignore"); import os; os.makedirs("/tmp/mod07", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)
NROW,NCOL=10,20; N=NROW*NCOL
row_idx=np.repeat(np.arange(NROW),NCOL); col_idx=np.tile(np.arange(NCOL),NROW)
lon=-120+col_idx*2.5+np.random.normal(0,0.3,N)
lat=28+row_idx*2.0+np.random.normal(0,0.2,N)
pct_poverty=10+8*np.random.beta(2,5,N)+3*np.sin(lon/20)
population=np.random.lognormal(10.5,1.0,N).astype(int)
def sp(vals,sigma=2):
    grid=np.zeros((NROW,NCOL))
    for r,c,v in zip(row_idx,col_idx,vals): grid[r,c]=v
    return gaussian_filter(grid,sigma=sigma)[row_idx,col_idx]
spatial_risk=sp(np.random.normal(0,1,N))
# Spatially patterned PM2.5 (higher in industrial areas — western region)
pm25_base = 8 + 4*(lon-lon.min())/(lon.max()-lon.min())  # W->E gradient
pm25 = pm25_base + sp(np.random.gamma(1,1,N), sigma=1.5) + np.random.normal(0,1,N)
pm25 = pm25.clip(3,30)
# True exposure-response: 1 ug/m3 PM2.5 -> 2.5 extra CVD deaths/100k
TRUE_PM25 = 2.5
cvd_rate = (160 + 1.2*pct_poverty + TRUE_PM25*pm25 + 10*spatial_risk + np.random.normal(0,10,N))
expected=(cvd_rate/100000)*population*5
observed=np.random.poisson(expected).astype(int)
smr=observed/expected.clip(1)
df=pd.DataFrame({"county_id":[f"C{i:03d}" for i in range(N)],
    "lon":lon,"lat":lat,"row":row_idx,"col":col_idx,
    "pct_poverty":pct_poverty,"pm25":pm25,"population":population,
    "cvd_rate":cvd_rate,"observed":observed,"expected":expected,"smr":smr})
print(f"N={N} | PM2.5 mean={pm25.mean():.1f} SD={pm25.std():.1f} | CVD={cvd_rate.mean():.1f}/100k")

## 2. Exposure Assessment — Interpolating Point Data to Areas

Air quality monitors are located at specific points but we need county-level estimates.

**Methods:**
| Method | Principle | Smoothness | Use case |
|--------|-----------|------------|----------|
| **IDW** | Weight = 1/distance^p | Moderate | Quick, no model |
| **Kriging (ordinary)** | Geostatistical, uses variogram | Smooth | Optimal linear prediction |
| **Nearest monitor** | Assign to closest monitor | Abrupt | Worst but common |
| **Land-use regression** | Predict from land use covariates | Flexible | Urban studies |


In [ ]:
from scipy.spatial import cKDTree
from scipy.interpolate import RBFInterpolator

# Simulate air quality monitoring stations (fewer than counties)
np.random.seed(13); N_MONITORS = 35
mon_lon = np.random.uniform(df.lon.min(), df.lon.max(), N_MONITORS)
mon_lat = np.random.uniform(df.lat.min(), df.lat.max(), N_MONITORS)
# Monitor readings match the spatial PM2.5 surface approximately
mon_pm25 = (8 + 4*(mon_lon-df.lon.min())/(df.lon.max()-df.lon.min())
            + np.random.gamma(1,1,N_MONITORS) + np.random.normal(0,1.5,N_MONITORS)).clip(3,28)

county_coords   = np.column_stack([df.lon, df.lat])
monitor_coords  = np.column_stack([mon_lon, mon_lat])

# Method 1: Nearest monitor assignment
tree = cKDTree(monitor_coords)
dists_nn, idx_nn = tree.query(county_coords, k=1)
df["pm25_nearest"] = mon_pm25[idx_nn]

# Method 2: Inverse Distance Weighting (IDW) - power p=2
def idw(query_pts, source_pts, values, p=2, k=8):
    tree = cKDTree(source_pts)
    dists, idxs = tree.query(query_pts, k=min(k, len(source_pts)))
    eps = 1e-10
    weights = 1.0 / (dists + eps)**p
    weights /= weights.sum(axis=1, keepdims=True)
    return (weights * values[idxs]).sum(axis=1)

df["pm25_idw"] = idw(county_coords, monitor_coords, mon_pm25, p=2, k=10)

# Method 3: Thin-plate spline (smooth interpolation)
try:
    rbf = RBFInterpolator(monitor_coords, mon_pm25, kernel="thin_plate_spline", smoothing=0.5)
    df["pm25_rbf"] = rbf(county_coords).clip(3, 30)
    RBF_OK = True
except Exception as e:
    df["pm25_rbf"] = df["pm25_idw"]
    RBF_OK = False
    print(f"RBF fallback to IDW: {e}")

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col, title in [
    (axes[0], "pm25",         "True PM2.5 surface"),
    (axes[1], "pm25_nearest", "Nearest monitor"),
    (axes[2], "pm25_idw",     "IDW (p=2, k=10)"),
    (axes[3], "pm25_rbf",     "Thin-plate spline RBF"),
]:
    vmin, vmax = df["pm25"].quantile(0.05), df["pm25"].quantile(0.95)
    sc = ax.scatter(df.lon, df.lat, c=df[col], cmap="YlOrBr",
                    vmin=vmin, vmax=vmax, s=120, alpha=0.85, edgecolors="white")
    ax.scatter(mon_lon, mon_lat, marker="^", c="blue", s=60, zorder=5, label="Monitor")
    plt.colorbar(sc, ax=ax, label="PM2.5")
    corr = np.corrcoef(df["pm25"], df[col])[0,1]
    ax.set_title(f"{title}
(r={corr:.3f} with truth)", fontsize=9)
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
plt.suptitle("PM2.5 Exposure Assessment: Interpolation Methods Comparison", fontsize=12)
plt.tight_layout(); plt.savefig("/tmp/mod07/pm25_interpolation.png", bbox_inches="tight"); plt.show()


## 3. Exposure-Response Analysis: Ecological Regression

In [ ]:
import statsmodels.api as sm

# Ecological regression: county CVD rate ~ county PM2.5
X = sm.add_constant(df[["pm25","pct_poverty"]])
model_eco = sm.OLS(df.cvd_rate, X).fit()
print("Ecological regression: CVD ~ PM2.5 + Poverty")
print(model_eco.summary().tables[1])
pm25_coef = model_eco.params["pm25"]
print(f"\nTrue PM2.5 coefficient: {TRUE_PM25:.1f} CVD deaths/100k per µg/m³")
print(f"Estimated coefficient : {pm25_coef:.1f}")
print(f"Bias                  : {pm25_coef - TRUE_PM25:+.2f}")

# Exposure-response curve (non-parametric)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df.pm25, df.cvd_rate, alpha=0.4, s=20, color="#4878CF")
x_line = np.linspace(df.pm25.min(), df.pm25.max(), 100)
y_pred = model_eco.params["Intercept"] + model_eco.params["pm25"]*x_line + model_eco.params["pct_poverty"]*df.pct_poverty.mean()
axes[0].plot(x_line, y_pred, "r-", lw=2, label=f"β={pm25_coef:.2f} (True: {TRUE_PM25})")
axes[0].set_xlabel("County PM2.5 (µg/m³)"); axes[0].set_ylabel("CVD rate per 100k")
axes[0].set_title("Exposure-Response: PM2.5 → CVD Mortality"); axes[0].legend(fontsize=9)

# Using different interpolation methods
for col, color, label in [("pm25_nearest","#D65F5F","Nearest"),
                            ("pm25_idw","#4878CF","IDW"),
                            ("pm25_rbf","#6ACC65","RBF")]:
    Xm = sm.add_constant(df[[col,"pct_poverty"]])
    m  = sm.OLS(df.cvd_rate, Xm).fit()
    coef = m.params[col]
    axes[1].bar(label, coef, color=color, alpha=0.85, width=0.6)
axes[1].axhline(TRUE_PM25, color="black", ls="--", lw=2, label=f"True β={TRUE_PM25}")
axes[1].set_ylabel("PM2.5 coefficient"); axes[1].set_title("Coefficient by Exposure Method
(exposure measurement error matters!)")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod07/exposure_response.png", bbox_inches="tight"); plt.show()


## 4. Environmental Justice Analysis

In [ ]:
# Environmental justice: is pollution burden distributed inequitably?
# Quintile analysis: compare PM2.5 exposure across racial/income groups

# Simulate demographic data
np.random.seed(42)
df["pct_minority"]    = (20 + 15*np.random.beta(2,3,N)).clip(0,100)
df["pct_low_income"]  = df.pct_poverty * 2.5 + np.random.normal(0,5,N)
df["pct_low_income"]  = df["pct_low_income"].clip(0,100)

# Environmental justice metrics
n_quint = 5
df["poverty_quintile"] = pd.qcut(df.pct_poverty, n_quint, labels=False) + 1
df["minority_quintile"] = pd.qcut(df.pct_minority, n_quint, labels=False) + 1

ej_poverty  = df.groupby("poverty_quintile")["pm25"].mean()
ej_minority = df.groupby("minority_quintile")["pm25"].mean()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# PM2.5 by poverty quintile
axes[0].bar(ej_poverty.index, ej_poverty.values, color=plt.cm.RdYlGn_r(np.linspace(0.2,0.9,5)),
            alpha=0.85, edgecolor="white")
axes[0].set_xlabel("Poverty quintile (1=lowest)"); axes[0].set_ylabel("Mean PM2.5 (µg/m³)")
axes[0].set_title(f"PM2.5 by Poverty Quintile
(EJ concern: slope={np.polyfit(range(1,6),ej_poverty.values,1)[0]:.2f})")

# PM2.5 by minority quintile
axes[1].bar(ej_minority.index, ej_minority.values, color=plt.cm.RdYlGn_r(np.linspace(0.2,0.9,5)),
            alpha=0.85, edgecolor="white")
axes[1].set_xlabel("% Minority quintile (1=lowest)"); axes[1].set_ylabel("Mean PM2.5 (µg/m³)")
axes[1].set_title(f"PM2.5 by Minority Share
(slope={np.polyfit(range(1,6),ej_minority.values,1)[0]:.2f})")

# Scatter: minority share vs PM2.5
axes[2].scatter(df.pct_minority, df.pm25, alpha=0.4, s=20, c=df.pct_poverty, cmap="RdYlGn_r")
m,b = np.polyfit(df.pct_minority, df.pm25, 1)
xl = np.linspace(df.pct_minority.min(), df.pct_minority.max(), 100)
axes[2].plot(xl, m*xl+b, "k-", lw=2, label=f"β={m:.3f}")
r = np.corrcoef(df.pct_minority, df.pm25)[0,1]
axes[2].set_xlabel("% Minority"); axes[2].set_ylabel("PM2.5 (µg/m³)")
axes[2].set_title(f"Minority Share vs PM2.5
(r={r:.3f})")
axes[2].legend(fontsize=9)
plt.suptitle("Environmental Justice Analysis: Disparities in Pollution Exposure", fontsize=12)
plt.tight_layout(); plt.savefig("/tmp/mod07/ej_analysis.png", bbox_inches="tight"); plt.show()

# Dissimilarity index (EJ metric)
pm25_high = (df.pm25 > df.pm25.median()).astype(int)
minority_high = (df.pct_minority > df.pct_minority.median()).astype(int)
assoc = np.corrcoef(df.pct_minority, df.pm25)[0,1]
print(f"Environmental Justice Summary:")
print(f"  Correlation: minority% vs PM2.5 = {assoc:.3f}")
print(f"  Highest poverty quintile PM2.5  : {ej_poverty.iloc[-1]:.1f} µg/m³")
print(f"  Lowest poverty quintile PM2.5   : {ej_poverty.iloc[0]:.1f} µg/m³")
print(f"  Inequality ratio (high/low Q)   : {ej_poverty.iloc[-1]/ej_poverty.iloc[0]:.2f}x")


## 5. Ecological Fallacy — Individual vs Aggregate Data

In [ ]:
# Simulate individual-level data to contrast with county-level estimates
np.random.seed(7)
N_IND = 10000

# Individual PM2.5 (based on county + within-county variation)
ind_county = np.random.choice(N, N_IND)  # assign each person to a county
ind_county_pm25 = df.pm25.values[ind_county]
ind_pm25 = ind_county_pm25 + np.random.normal(0, 3, N_IND)  # within-county variation
ind_poverty = df.pct_poverty.values[ind_county] + np.random.normal(0, 5, N_IND)
ind_age     = np.random.normal(55, 15, N_IND).clip(20, 90)

# Individual CVD risk (log-odds)
TRUE_IND_PM25 = 0.015  # log-OR per ug/m3 at individual level
ind_logit = (-3.5 + TRUE_IND_PM25*ind_pm25 + 0.02*(ind_age-55)
             + 0.015*(ind_poverty-10) + np.random.normal(0, 0.5, N_IND))
ind_cvd = np.random.binomial(1, 1/(1+np.exp(-ind_logit)), N_IND)

# Ecological estimate
eco_coef_log_rr = np.polyfit(df.pm25, np.log(df.cvd_rate), 1)[0]
ind_coef_log_or = sm.Logit(ind_cvd, sm.add_constant(pd.DataFrame({"pm25":ind_pm25,"age":ind_age,"poverty":ind_poverty}))).fit(disp=0).params["pm25"]

print("Ecological Fallacy Demonstration:")
print(f"  True individual log-OR per µg/m³ PM2.5: {TRUE_IND_PM25:.4f}")
print(f"  Individual-level logistic estimate     : {ind_coef_log_or:.4f}")
print(f"  County-level ecological estimate       : {eco_coef_log_rr:.4f}")
print()
print("The ecological and individual-level estimates are on different scales")
print("(rate ratio vs odds ratio) and answer different questions.")
print("DO NOT use county-level coefficients to make individual-level inferences!")


## 6. Knowledge Check
1. IDW gives PM2.5 coefficient β=2.1 while RBF gives β=2.8. Which should you prefer and why?
2. PM2.5 exposure and poverty are correlated (r=0.41). How does this affect the ecological regression for PM2.5's health effect?
3. The EJ analysis shows highest-poverty quintile has 1.4x higher PM2.5 than lowest. Is this statistically significant? How would you test it?
4. Individual-level PM2.5 log-OR = 0.015 vs ecological log-RR = 0.07. Why do these differ? What is the ecological fallacy specifically?
5. Implement Moran's I for PM2.5 exposure to quantify spatial patterning of the exposure surface.


In [ ]:
# Exercise 5 — Moran's I for PM2.5 exposure
def build_queen_rs(row_idx,col_idx,nrow,ncol):
    N=len(row_idx); W=np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            if i!=j and max(abs(row_idx[i]-row_idx[j]),abs(col_idx[i]-col_idx[j]))<=1:
                W[i,j]=1
    rs=W.sum(axis=1,keepdims=True); rs[rs==0]=1; return W/rs
W=build_queen_rs(df.row.values,df.col.values,NROW,NCOL)
def morans_i(y,W,n_perm=499,seed=42):
    n=len(y); z=y-y.mean(); S0=W.sum()
    I_obs=(n/S0)*(z@(W@z))/(z@z)
    rng=np.random.default_rng(seed)
    I_perm=[(n/S0)*(rng.permutation(z)@(W@rng.permutation(z)))/(z@z) for _ in range(n_perm)]
    p=(np.abs(I_perm)>=abs(I_obs)).mean()
    return I_obs,p

I_pm25, p_pm25 = morans_i(df.pm25.values, W)
I_cvd,  p_cvd  = morans_i(df.cvd_rate.values, W)
print(f"Moran's I for PM2.5    : {I_pm25:.4f} (p={p_pm25:.4f})")
print(f"Moran's I for CVD rate : {I_cvd:.4f}  (p={p_cvd:.4f})")
print()
print("Both exposure and outcome are spatially autocorrelated.")
print("This means standard SE in ecological regression are too small -> need spatial model!")


---
**Next → NB-07: Health Service Area Analysis & Access Metrics**